# 01 · MD-readiness scoring: Assemblies


**Supplementary material to BioExcel-3 Deliverable D2.5**, *Final report on standards and recommendations for simulation, data generation, storage and mining*.


Part 1 of 3: **01 assemblies** → 02 non-assemblies → 03 analysis (figures and tables).


**What this notebook does**
1. Loads per-assembly issue counts from biobb_structure_checking results
2. Computes the MD-readiness penalty 
3. Assigns remediation tiers 
4. Ranks assemblies within each PDB entry


**How to run:** *Restart Kernel and Run All*. If MongoDB is reachable (add credentials in `.env`), the counts are extracted from the database and cached. If it is not reachable, the parquet cache is used.


**Contents**
1. [Setup](#sec1)
2. [Severity weights and remediation classes](#sec2)
3. [MongoDB extraction](#sec3)
4. [Scoring, tiering and ranking](#sec4)
5. [Load data](#sec5)
6. [Rank and save](#sec6)
7. [Example entries](#sec7)
   - [7.1 Ranking within selected PDB entries](#sec7-1)
   - [7.2 Single-assembly detail](#sec7-2)
8. [Environment](#sec8)

<a id="sec1"></a>
## 1. Setup

Imports, file paths and the run option.

In [1]:
import json
import os
import platform
import re
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from pymongo import MongoClient
from pymongo.errors import ConnectionFailure, OperationFailure
from tabulate import tabulate
import time
from datetime import datetime

# --- Paths -------------------------------------------------------------------
ISSUE_COUNTS_CACHE_PATH = Path("data/assembly_issue_counts.parquet")    # raw counts (MongoDB extraction)
RANKED_CACHE_PATH = Path("data/assembly_md_readiness_ranked.parquet")  # scored and ranked output
ISSUE_CONFIG_PATH = Path("data/assembly_issue_config.json")             # weights/classes used (read by 03)

# --- Run option ----------------------------------------------------------------
USE_CACHE_ONLY = False   # True: skip MongoDB and read the cache directly

<a id="sec2"></a>
## 2. Severity weights and remediation classes

Each issue has two independent properties:
- a **severity weight**, which sets its contribution to the penalty;
- a **remediation class**, which sets the tier.

In [2]:
# =============================================================================
# 2.1 Issue configuration (issue numbers follow the D2.4 classification)
#
# Weights:  Mild = 1 | Mild–Medium = 2 | Medium = 3 | Medium–Severe = 4 | Severe = 5
# Classes:  system_modification - system preparation; macromolecule not rebuilt
#           structural_repair   - well-defined structural correction
#           major_defect        - substantial or uncertain reconstruction
# =============================================================================

ASM_ISSUE_CONFIG = {

    # --- Experimental structure ambiguity ---
    "num_models":               {"weight": 1, "remediation": "system_modification"},  # Mild - Issue 1: Multiple models
    "altloc_count":             {"weight": 3, "remediation": "system_modification"},  # Medium - Issue 5: Alternate locations
    "sequence_mismatch_count":  {"weight": 3, "remediation": "system_modification"},  # Medium - Issue 5: Microheterogeneity

    # --- System / composition ---
    "num_ligands":              {"weight": 5, "remediation": "system_modification"},  # Severe - Issue 4: Ligands / non-standard residues

    # --- Structural completeness ---
    "sidechain_missing_count":  {"weight": 1, "remediation": "structural_repair"},    # Mild - Issue 6: Missing side-chain atoms
    "missing_atoms_count":      {"weight": 1, "remediation": "structural_repair"},    # Mild - Issue 6: Missing backbone atoms
    "backbone_breaks_count":    {"weight": 5, "remediation": "major_defect"},         # Severe - Issue 6: Backbone breaks

    # --- Chemical / topology validity ---
    "sidechain_unknown_count":  {"weight": 5, "remediation": "major_defect"},         # Severe - Issue 8: Unknown atoms

    # --- Stereochemistry / conformation ---
    "side_chirals_count":       {"weight": 1, "remediation": "structural_repair"},    # Mild - Issue 9: Side-chain chirality
    "bck_chirals_count":        {"weight": 5, "remediation": "structural_repair"},    # Severe - Issue 9: Backbone chirality
    "cis_nonpro_count":         {"weight": 5, "remediation": "structural_repair"},    # Severe - Issue 10: Incorrect cis peptides (non X-Pro only)  
    "unusual_trans_count":      {"weight": 3, "remediation": "structural_repair"},    # Medium - Issue 10: non-planar trans peptide bond    
    "amide_count":              {"weight": 2, "remediation": "structural_repair"},    # Mild-Medium - Issue 11: Amide orientation

    # --- Chemical state / hydrogens ---
    "num_ionizable":            {"weight": 1, "remediation": "system_modification"},  # Mild - Issue 12: Protonation (excl. His)
    "num_histidines_ionizable": {"weight": 2, "remediation": "system_modification"},  # Mild-Medium - Issue 12: Histidine state
    "deuterium_residue_count":  {"weight": 1, "remediation": "system_modification"},  # Mild - Issue 8: Deuterium-labelled residues

    # --- Covalent structure ---
    "disulfide_bnds":           {"weight": 1, "remediation": "system_modification"},  # Mild - Issue 13: Disulfide bridges

    # --- Structural water ---
    "contacting_water_count":   {"weight": 5, "remediation": "system_modification"},  # Severe - Issue 14: Structural water

    # --- Geometry ---
    "severe_clashes_count":     {"weight": 5, "remediation": "structural_repair"},    # Severe - Issue 15: Severe clashes
    "apolar_clashes_count":     {"weight": 2, "remediation": "structural_repair"},    # Mild-Medium - Issue 15: Apolar clashes
}

In [3]:
# =============================================================================
# 2.2 Lookup tables derived from the configuration
# =============================================================================

# Weights -> penalty
ASM_ISSUE_WEIGHTS = {f: cfg["weight"] for f, cfg in ASM_ISSUE_CONFIG.items()}
ASM_WEIGHTS_SERIES = pd.Series(ASM_ISSUE_WEIGHTS)

# Remediation classes -> tier
SYSTEM_MODIFICATION_FIELDS = [f for f, cfg in ASM_ISSUE_CONFIG.items() if cfg["remediation"] == "system_modification"]
STRUCTURAL_REPAIR_FIELDS   = [f for f, cfg in ASM_ISSUE_CONFIG.items() if cfg["remediation"] == "structural_repair"]
MAJOR_DEFECT_FIELDS        = [f for f, cfg in ASM_ISSUE_CONFIG.items() if cfg["remediation"] == "major_defect"]

# Size basis: residue-based issues per 100 residues, atom-based issues per 100 atoms
RESIDUE_BASIS_FIELDS = [
    "altloc_count", "sidechain_missing_count", "missing_atoms_count",
    "backbone_breaks_count", "sidechain_unknown_count", "side_chirals_count",
    "bck_chirals_count", "cis_nonpro_count", "amide_count", "unusual_trans_count",
    "disulfide_bnds", "num_ligands", "num_ionizable", "num_histidines_ionizable", "deuterium_residue_count",
    "sequence_mismatch_count",
]
ATOM_BASIS_FIELDS = ["severe_clashes_count", "apolar_clashes_count", "contacting_water_count"]

# num_models cannot be normalised by size: flat penalty when > 1 model
MODELS_FLAG_WEIGHT = ASM_ISSUE_WEIGHTS["num_models"]

# Check: every weighted field has a size basis, and vice versa
BASIS_FIELDS = set(RESIDUE_BASIS_FIELDS) | set(ATOM_BASIS_FIELDS)
WEIGHT_FIELDS = set(ASM_ISSUE_WEIGHTS) - {"num_models"}
assert BASIS_FIELDS == WEIGHT_FIELDS, (
    f"Basis fields missing from weights: {sorted(BASIS_FIELDS - WEIGHT_FIELDS)}; "
    f"Weights missing from basis fields: {sorted(WEIGHT_FIELDS - BASIS_FIELDS)}"
)

<a id="sec3"></a>
## 3. MongoDB extraction

Connection, classification helpers, and the aggregation that gives one row per assembly. Missing fields count as zero.

In [4]:
# =============================================================================
# 3.1 Connection (read-only, credentials from .env)
# =============================================================================

def connect_to_db():
    """
    Connect to MongoDB and return the PDB checking collection, or None if unavailable.
    """
    load_dotenv()

    mongoHost = os.getenv('MONGO_HOST', default='localhost')     # MongoDB host
    mongoPort = os.getenv('MONGO_PORT', default='27018')         # port (often an SSH tunnel)
    mongoUser = os.getenv('MONGO_USER')                          # username
    mongoPass = os.getenv('MONGO_PASS')                          # password
    mongoAuthSrc = os.getenv('MONGO_AUTH_SRC', default='admin')  # authentication DB
    mongoDb = os.getenv('MONGO_DB', default='FlexPortal')        # target database (read only)

    try:
        mongoClient = MongoClient(
            host=mongoHost,
            port=int(mongoPort),
            username=mongoUser,
            password=mongoPass,
            authSource=mongoAuthSrc,
            serverSelectionTimeoutMS=5000  # give up after 5 s if the server is down
        )
        mongoClient.admin.command('ping')  # force a connection check now

        db = mongoClient[mongoDb]
        collection = db["PDB_Checking"]
        return collection

    except (ConnectionFailure, OperationFailure) as e:
        print(f"MongoDB not available ({type(e).__name__}) -> using parquet cache")
        return None

In [5]:
# =============================================================================
# 3.2 Classification helpers
# =============================================================================

# Metal-ion codes used to classify ligands.detected (the field also lists non-metals, e.g. ATP)
KNOWN_METAL_CODES = [
    "MG", "ZN", "CA", "MN", "FE", "FE2", "CU", "CU1", "NA", "K", "CO", "NI",
    "CD", "HG", "PT", "AG", "AU", "LI", "BA", "SR", "CS", "AL", "CR", "MO", "W",
]

# Standard amino acids, used to identify deuterium-labelled residues
STANDARD_AA_CODES = [
    "ALA", "ARG", "ASN", "ASP", "CYS", "GLN", "GLU", "GLY", "HIS", "ILE",
    "LEU", "LYS", "MET", "PHE", "PRO", "SER", "THR", "TRP", "TYR", "VAL",
]

# Deuterium residue: a standard amino acid whose unknown atoms all follow
# the D-for-H naming pattern (e.g. "DD21" for "HD21")
_deuterium_residue_check = {
    "$let": {
        "vars": {
            # Residue type from the key (e.g. "ALA A 10" -> "ALA")
            "res_type": {
                "$arrayElemAt": [
                    {"$split": ["$$res.k", " "]},
                    0
                ]
            }
        },
        "in": {
            "$and": [
                # Standard amino acids only
                {"$in": ["$$res_type", STANDARD_AA_CODES]},
                {
                    # Every unknown atom name matches the D-for-H pattern
                    "$allElementsTrue": {
                        "$map": {
                            "input": "$$res.v",
                            "as": "atom",
                            "in": {
                                "$regexMatch": {
                                    "input": "$$atom",
                                    "regex": "^D[A-Z0-9]*$"
                                }
                            }
                        }
                    }
                },
            ]
        }
    }
}

In [6]:
# =============================================================================
# 3.3 Extraction: one row per assembly
# =============================================================================

def get_assembly_issue_counts(collection, filter_query=None):

    """
    Extract raw structural-quality metrics, one row per numbered assembly.

    AlphaFold models are excluded and assembly identifiers must end in
    '_<number>'. No scoring or normalization is performed here.

    Parameters
    ----------
    collection : pymongo.collection.Collection
        MongoDB collection containing PDB checking records.
    filter_query : dict, optional
        Additional MongoDB filter applied before assembly selection.

    Returns
    -------
    pandas.DataFrame
        One row per assembly with raw issue counts and structure metadata.
    """

    pipeline = []
    if filter_query:
        pipeline.append({"$match": filter_query})

    pipeline += [
        {"$match": {"_id": {"$not": re.compile(r"^AF-")}}},         # exclude AlphaFold
        {"$match": {"_id": re.compile(r"_\d+$")}},                  # numbered assemblies only (any _N)

        {"$project": {

            # --- Identity and size ---
            "assembly_id": "$_id",
            "experimental_method": "$headers.method",
            "residue_count": "$stats.stats.num_res",
            "atoms_count": "$stats.stats.num_ats",
            "num_models": "$models.nmodels",

            # --- Defect counts (missing fields = zero) ---
            "backbone_breaks_count": {"$size": {"$ifNull": ["$backbone.breaks.detected", []]}},
            "missing_atoms_count": {
                "$cond": [
                    {"$eq": [{"$type": "$backbone.missing_atoms"}, "object"]},
                    {"$size": {"$objectToArray": "$backbone.missing_atoms"}}, 0
                ]
            },
            "sidechain_missing_count": {
                "$cond": [
                    {"$eq": [{"$type": "$fixside.detected_missing"}, "object"]},
                    {"$size": {"$objectToArray": "$fixside.detected_missing"}}, 0
                ]
            },

            "side_chirals_count": {"$size": {"$ifNull": ["$chiral.detected", []]}},
            "bck_chirals_count": {"$size": {"$ifNull": ["$chiral_bck.detected", []]}},
            "severe_clashes_count": {"$size": {"$ifNull": ["$clashes.detected.severe", []]}},
            "apolar_clashes_count": {"$size": {"$ifNull": ["$clashes.detected.apolar", []]}},
            "amide_count": {"$size": {"$ifNull": ["$amide.detected", []]}},
            "num_ligands": {"$size": {"$ifNull": ["$ligands.detected", []]}},
            "cis_pro_count": {
                "$cond": [
                    {"$eq": [{"$type": "$cistransbck"}, "object"]},
                    {"$size": {"$filter": {
                        "input": {"$ifNull": ["$cistransbck.cis", []]},
                        "as": "b",
                        "cond": {"$regexMatch": {"input": {"$arrayElemAt": ["$$b", 1]}, "regex": "^(PRO|HYP) "}},
                    }}},
                    0,
                ]
            },
            "cis_nonpro_count": {
                "$cond": [
                    {"$eq": [{"$type": "$cistransbck"}, "object"]},
                    {"$size": {"$filter": {
                        "input": {"$ifNull": ["$cistransbck.cis", []]},
                        "as": "b",
                        "cond": {"$not": {"$regexMatch": {"input": {"$arrayElemAt": ["$$b", 1]}, "regex": "^(PRO|HYP) "}}},
                    }}},
                    0,
                ]
            },
            "unusual_trans_count": {
                "$cond": [
                    {"$eq": [{"$type": "$cistransbck"}, "object"]},
                    {"$size": {"$ifNull": ["$cistransbck.unusual_trans", []]}},
                    0,
                ]
            },            
            "altloc_count": {
                "$size": {
                    "$filter": {
                        "input": {"$objectToArray": {"$ifNull": ["$altloc", {}]}},
                        "as": "item",
                        "cond": {"$not": {"$in": ["$$item.k", ["selected", "opts", "error"]]}}
                    }
                }
            },

            # --- Microheterogeneity: residue-level mismatches over all chains ---
            "sequence_mismatch_count": {
                "$cond": [
                    {"$eq": [{"$type": "$sequence_mismatches"}, "object"]},
                    {
                        "$sum": {
                            "$map": {
                                "input": {"$objectToArray": "$sequence_mismatches"},
                                "as": "chain",
                                "in": {"$size": {"$ifNull": ["$$chain.v.mismatches", []]}}
                            }
                        }
                    },
                    0
                ]
            },

            # --- Disulfide bridges ---
            "disulfide_bnds": {
                "$switch": {
                    "branches": [
                        {"case": {"$eq": [{"$type": "$getss.found"}, "array"]}, "then": {"$size": "$getss.found"}}, # {"found": [...]} -> bonds detected
                        {"case": {"$eq": [{"$type": "$getss"}, "array"]}, "then": {"$size": "$getss"}},             # []               -> no bonds
                    ],
                    "default": 0
                }
            },

            # --- Waters in contact with the structure (isolated waters excluded) ---
            "contacting_water_count": {
                "$size": {
                    "$filter": {
                        "input": {"$objectToArray": {"$ifNull": ["$water.contacts", {}]}},
                        "as": "w",
                        "cond": {"$gt": [{"$size": "$$w.v"}, 0]}
                    }
                }
            },

            # --- Ionizable residues: His counted separately (no double counting) ---
            "num_ionizable": {
                "$size": {
                    "$filter": {
                        "input": {"$ifNull": ["$add_hydrogen.ionic_detected", []]},
                        "as": "res",
                        "cond": {"$not": {"$regexMatch": {"input": "$$res", "regex": "^HIS"}}}   # excludes His
                    }
                }
            },
            "num_histidines_ionizable": {
                "$size": {
                    "$filter": {
                        "input": {"$ifNull": ["$add_hydrogen.ionic_detected", []]},
                        "as": "res",
                        "cond": {"$regexMatch": {"input": "$$res", "regex": "^HIS"}}
                    }
                }
            },

            # --- Issue 4 split (tier only; the penalty uses num_ligands once) ---
            "nonstandard_residue_count": {"$size": {"$ifNull": ["$backbone.mod_residues", []]}},
            "metal_ion_count": {
                "$size": {
                    "$filter": {
                        "input": {"$ifNull": ["$ligands.detected", []]},
                        "as": "l",
                        "cond": {
                            "$in": [
                                {"$arrayElemAt": [{"$split": [{"$trim": {"input": "$$l"}}, " "]}, 0]},
                                KNOWN_METAL_CODES
                            ]
                        }
                    }
                }
            },
            "small_molecule_ligand_count": {
                "$size": {
                    "$filter": {
                        "input": {"$ifNull": ["$ligands.detected", []]},
                        "as": "l",
                        "cond": {
                            "$and": [
                                {"$not": {"$in": ["$$l", {"$ifNull": ["$backbone.mod_residues", []]}]}},
                                {"$not": {"$in": [
                                    {"$arrayElemAt": [{"$split": [{"$trim": {"input": "$$l"}}, " "]}, 0]},
                                    KNOWN_METAL_CODES
                                ]}}
                            ]
                        }
                    }
                }
            },

            # --- Issue 8: genuine unknown atoms vs deuterium-labelled residues ---
            "sidechain_unknown_count": {
                "$size": {
                    "$filter": {
                        "input": {"$objectToArray": {"$ifNull": ["$fixside.detected_unknown", {}]}},
                        "as": "res",
                        "cond": {"$not": _deuterium_residue_check},   # genuine unknowns only
                    }
                }
            },
            "deuterium_residue_count": {
                "$size": {
                    "$filter": {
                        "input": {"$objectToArray": {"$ifNull": ["$fixside.detected_unknown", {}]}},
                        "as": "res",
                        "cond": _deuterium_residue_check,
                    }
                }
            },
        }},

        # --- base_id groups the assemblies of one PDB entry (e.g. "108M_1", "108M_2" -> "108M") ---
        {"$addFields": {"base_id": {"$arrayElemAt": [{"$split": ["$assembly_id", "_"]}, 0]}}},
    ]

    df = pd.DataFrame(collection.aggregate(pipeline, allowDiskUse=True, batchSize=10000))
    if df.empty:
        return df

    df["residue_count"] = pd.to_numeric(df["residue_count"], errors="coerce")
    df["atoms_count"] = pd.to_numeric(df["atoms_count"], errors="coerce")
    df["num_models"] = pd.to_numeric(df["num_models"], errors="coerce")
    return df

<a id="sec4"></a>
## 4. Scoring, tiering and ranking

- **Penalty:** issue counts per 100 residues or 100 atoms, multiplied by their weights and summed.
- **Tier:** set by the most demanding remediation class present, independent of the penalty.
- **Rank:** within each PDB entry, ordered by fully scored first, then tier, then penalty.

In [7]:
# =============================================================================
# 4. Scoring, remediation tiering and ranking
#
# Tier (highest intervention wins):
#   1. major_defect         - substantial/uncertain reconstruction
#   2. structural_repair    - constrained structural correction
#   3. system_modification  - preparation/parameterisation decision
#   4. MD-ready as deposited
# Unscored: missing size data AND no defects detected.
# =============================================================================

def rank_assemblies(df):

    """
    Calculate MD-readiness penalties, assign remediation tiers, and rank
    assemblies within each base_id family.
    """

    df = df.copy()

    # ------------------------------------------------------------------
    # SCORE
    # ------------------------------------------------------------------

    # 4a. Valid denominators (non-positive or missing -> NaN)
    residue_denom = df["residue_count"].where(df["residue_count"] > 0)
    atom_denom = df["atoms_count"].where(df["atoms_count"] > 0)

    # 4b. Residue-basis penalty: rate per 100 residues x weight
    residue_rate = df[RESIDUE_BASIS_FIELDS].div(residue_denom, axis=0) * 100
    residue_penalty = residue_rate.mul(
        ASM_WEIGHTS_SERIES[RESIDUE_BASIS_FIELDS], axis=1
    ).sum(axis=1, min_count=1)

    # 4c. Atom-basis penalty: rate per 100 atoms x weight
    atom_rate = df[ATOM_BASIS_FIELDS].div(atom_denom, axis=0) * 100
    atom_penalty = atom_rate.mul(
        ASM_WEIGHTS_SERIES[ATOM_BASIS_FIELDS], axis=1
    ).sum(axis=1, min_count=1)

    # 4d. Multiple-models flag (missing num_models -> single model)
    df["has_multiple_models"] = df["num_models"].fillna(1) > 1
    models_penalty = df["has_multiple_models"].astype(int) * MODELS_FLAG_WEIGHT

    # 4e. Total penalty (NaN if either denominator is missing -> not fully scored)
    df["md_readiness_penalty"] = residue_penalty + atom_penalty + models_penalty
    df["fully_scored"] = df["md_readiness_penalty"].notna()

    # ------------------------------------------------------------------
    # TIER
    # ------------------------------------------------------------------

    # 4f. Highest required intervention takes precedence
    has_major = (df[MAJOR_DEFECT_FIELDS].fillna(0) > 0).any(axis=1)

    has_repair = (
        (df[STRUCTURAL_REPAIR_FIELDS].fillna(0) > 0).any(axis=1)
        | (df["nonstandard_residue_count"].fillna(0) > 0)          # modified residues -> repair
    )

    # Metal ions and small molecules -> system modification (num_ligands used for the penalty only)
    other_modification_fields = [f for f in SYSTEM_MODIFICATION_FIELDS if f not in ("num_ligands", "num_models")]
    has_modification = (
        (df["metal_ion_count"].fillna(0) > 0)
        | (df["small_molecule_ligand_count"].fillna(0) > 0)
        | df["has_multiple_models"]
        | (df[other_modification_fields].fillna(0) > 0).any(axis=1)
    )

    df["tier"] = np.select(
        [has_major, has_repair, has_modification],
        ["Not MD-ready (major defect)", "Needs structural repair", "MD-ready with system modification"],
        default="MD-ready as deposited",
    )

    # Incomplete score and no defects detected -> possibly unprocessed -> Unscored
    all_raw_fields = RESIDUE_BASIS_FIELDS + ATOM_BASIS_FIELDS + [
        "nonstandard_residue_count", "metal_ion_count", "small_molecule_ligand_count",
        "cis_pro_count",
    ]
    appears_unprocessed = (df[all_raw_fields].fillna(0) == 0).all(axis=1) & ~df["fully_scored"]
    df.loc[appears_unprocessed, "tier"] = "Unscored (missing size data)"

    # ------------------------------------------------------------------
    # RANK
    # ------------------------------------------------------------------

    # 4g. Within each base_id: scored first -> tier -> penalty (lower = better)
    tier_order = {
        "MD-ready as deposited": 0,
        "MD-ready with system modification": 1,
        "Needs structural repair": 2,
        "Not MD-ready (major defect)": 3,
        "Unscored (missing size data)": 4,
    }
    df["_tier_rank"] = df["tier"].map(tier_order)

    df = df.sort_values(
        ["base_id", "fully_scored", "_tier_rank", "md_readiness_penalty"],
        ascending=[True, False, True, True]
    )
    df["rank_within_structure"] = df.groupby("base_id").cumcount() + 1

    return df.drop(columns="_tier_rank")

<a id="sec5"></a>
## 5. Load data

The notebook first tries MongoDB: if the connection succeeds, it extracts the counts and refreshes the cache. If MongoDB is unavailable (or `USE_CACHE_ONLY = True`), it reads the parquet cache.

In [8]:
import time
from datetime import datetime

# =============================================================================
# 5. MongoDB -> parquet cache fallback
# =============================================================================

collection = None if USE_CACHE_ONLY else connect_to_db()

if collection is not None:
    print(
        f"[INFO] Starting get_assembly_issue_counts() at "
        f"{datetime.now():%Y-%m-%d %H:%M:%S}"
    )

    start = time.perf_counter()

    df = get_assembly_issue_counts(collection)

    elapsed = time.perf_counter() - start

    print(f"[INFO] Finished at {datetime.now():%Y-%m-%d %H:%M:%S}")
    print(
        f"[TIMING] get_assembly_issue_counts() took "
        f"{elapsed:.2f} seconds ({elapsed / 60:.2f} minutes)"
    )

    ISSUE_COUNTS_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(ISSUE_COUNTS_CACHE_PATH)
    source = "MongoDB (cache refreshed)"

elif ISSUE_COUNTS_CACHE_PATH.exists():
    df = pd.read_parquet(ISSUE_COUNTS_CACHE_PATH)
    source = "parquet cache"

else:
    raise RuntimeError(
        f"No MongoDB connection and no cache at {ISSUE_COUNTS_CACHE_PATH}"
    )

print(f"Source: {source} | Shape: {df.shape}")


[INFO] Starting get_assembly_issue_counts() at 2026-10-28 10:38:07
[INFO] Finished at 2026-10-28 10:42:17
[TIMING] get_assembly_issue_counts() took 250.68 seconds (4.18 minutes)
Source: MongoDB (cache refreshed) | Shape: (297328, 30)


<a id="sec6"></a>
## 6. Rank and save

Saves the ranked table and the issue configuration. Notebook 03 reads both, so it uses the same weights as this scoring.

In [9]:
ranked = rank_assemblies(df)
ranked.to_parquet(RANKED_CACHE_PATH)

with open(ISSUE_CONFIG_PATH, "w") as fh:
    json.dump(ASM_ISSUE_CONFIG, fh, indent=2)

print(f"Saved: {RANKED_CACHE_PATH}, {ISSUE_CONFIG_PATH}")

Saved: data/assembly_md_readiness_ranked.parquet, data/assembly_issue_config.json


<a id="sec7"></a>
## 7. Example entries

Summary figures and tables for both structure types are produced in notebook 03.

<a id="sec7-1"></a>
### 7.1 Ranking within selected PDB entries

- **3FAD / 1HV4:** same tier, ranked by penalty.
- **2H50:** identical assemblies, so the penalties tie.
- **3INS:** neutron structure. Deuterium residues are counted as a mild system modification, not as unknown atoms.

In [10]:
ids = ["3FAD", "1HV4", "2H50", "3INS"]

for base_id in ids:
    print(f"\n=== {base_id} ===")
    print(
        tabulate(
            ranked[ranked["base_id"] == base_id][
                ["assembly_id", "md_readiness_penalty", "tier", "rank_within_structure", "deuterium_residue_count", "num_models"]
            ],
            headers="keys",
            tablefmt="psql",
            showindex=False,
        )
    )


=== 3FAD ===
+---------------+------------------------+-------------------------+-------------------------+---------------------------+--------------+
| assembly_id   |   md_readiness_penalty | tier                    |   rank_within_structure |   deuterium_residue_count |   num_models |
|---------------+------------------------+-------------------------+-------------------------+---------------------------+--------------|
| 3FAD_1        |                86.5999 | Needs structural repair |                       1 |                         0 |            1 |
| 3FAD_2        |                87.2518 | Needs structural repair |                       2 |                         0 |            1 |
+---------------+------------------------+-------------------------+-------------------------+---------------------------+--------------+

=== 1HV4 ===
+---------------+------------------------+-------------------------+-------------------------+---------------------------+--------------+
| asse

<a id="sec7-2"></a>
### 7.2 Single-assembly detail

- **3INS_1:** needs structural repair (clashes, amide); its penalty is dominated by ligands.
- **100D_1:** system modification only; its penalty comes mainly from contacting waters.

In [11]:
for assembly_id in ["3INS_1", "100D_1"]:
    row = ranked.loc[ranked["assembly_id"] == assembly_id]
    if not row.empty:
        print(row.iloc[0].to_string(), "\n")

_id                                                            3INS_1
assembly_id                                                    3INS_1
experimental_method            X-RAY DIFFRACTION, NEUTRON DIFFRACTION
residue_count                                                     133
atoms_count                                                       865
num_models                                                          1
backbone_breaks_count                                               0
missing_atoms_count                                                 0
sidechain_missing_count                                             0
side_chirals_count                                                  0
bck_chirals_count                                                   0
severe_clashes_count                                               19
apolar_clashes_count                                               18
amide_count                                                         1
num_ligands         

<a id="sec8"></a>
## 8. Environment

Package versions used for this run.

In [12]:
import pymongo, pyarrow
print(f"Python {platform.python_version()} | numpy {np.__version__} | pandas {pd.__version__} | "
      f"pyarrow {pyarrow.__version__} | pymongo {pymongo.__version__}")

Python 3.11.15 | numpy 2.4.3 | pandas 3.0.1 | pyarrow 23.0.1 | pymongo 4.16.0
